In [4]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
import cv2
images=[]
masks=[]
image_path="Dataset_2/images"
mask_path="Dataset_2/masks"
image_files= sorted(os.listdir(image_path)) #important 
mask_files= sorted(os.listdir(mask_path)) 
for filename in image_files:
    path=os.path.join(image_path, filename)
    img=cv2.imread(path)
    img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img=cv2.resize(img, (256, 256))
    img=img.astype('float32')/255.0
    images.append(img)

for filename in mask_files:
    path=os.path.join(mask_path, filename)
    mask=cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    mask=cv2.resize(mask, (256, 256))
    mask=(mask > 127).astype('float32')
    mask=np.expand_dims(mask, axis=-1)
    masks.append(mask)
X=np.array(images)
y=np.array(masks)

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.15, random_state=42)

MedicAI is a Keras based library. We use MedicAI to employ a SwinUNet model included in the library. MedicAI also includes defined metrics that we used in the previous notebooks as well like the Dice score and the Dice loss. 

In [3]:
from medicai.models import SwinUNETR
from medicai.losses import BinaryDiceCELoss
from medicai.metrics import BinaryDiceMetric
model=SwinUNETR(input_shape=(256,256,3), num_classes=1)

In [7]:
model.compile(optimizer='adam', 
loss=BinaryDiceCELoss(from_logits=True,num_classes=1), 
metrics=[BinaryDiceMetric(from_logits=True,num_classes=1)])

In [ ]:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=3, batch_size=16)

Epoch 1/20
 24/163 ━━━━━━━━━━━━━━━━━━━━ 1:41:30 44s/step - base_dice: 0.1378 - loss: 1.0987